# 120. 과거센서만 사용하는 국소관계 - 이력 누락 보완
119번은 모든라벨 센서이력을 제거해574중111/미성형8중1개만20개이력이 충분했다. 이번에는 센서 이력 자체는 보존하되 과거 정상/불량 정답은 교사에서 완전히 제거한다. 현재/미래 센서는 예상관계 학습에서 제외. 평가구간 내부의 과거센서도 온라인적응에 쓸 수 있어 전체 평가구간의 센서통계를 배제한 검증과 다르다. 품질분류·대체·임계값·조합은 외부평가구간라벨 제외 유지.
원래라벨 Shot 센서와 비라벨 이력 통합/동일Timestamp는 기존 검증라벨 센서우선. RG3 동일설비만. 실제제품전환을 완전히복원한모델이 아니라30분공백근사. 공식테스트이력제공미확인. 이미본자료개발평가이며신규독립자료아님.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
labels=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=labels[labels['product'].eq('RG3')&(labels.PassOrFail.eq(0)|labels.Reason.str.contains('미성형',na=False))].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int);g=d.comparison_block.to_numpy();blocks=np.unique(g)
assert len(d)==574 and y.sum()==8
u=pd.read_csv(root/'data/original_split/moldset_unlabeled_rg3.csv',parse_dates=['TimeStamp'])
u=u[u.EQUIP_CD.isin(d.EQUIP_CD.unique())].sort_values('TimeStamp').drop_duplicates(['EQUIP_CD','TimeStamp'])
sensor_columns=['EQUIP_CD','TimeStamp','Max_Injection_Pressure','Max_Switch_Over_Pressure','Max_Injection_Speed','Filling_Time','Mold_Temperature_3','Plasticizing_Position','Cushion_Position','Average_Back_Pressure','Barrel_Temperature_5','Plasticizing_Time','Max_Screw_RPM']
# Keep past sensor observations but never pass their quality labels to the teacher.
label_sensors=labels[labels['product'].eq('RG3')][sensor_columns].copy()
u=pd.concat([label_sensors,u[sensor_columns]],ignore_index=True).drop_duplicates(['EQUIP_CD','TimeStamp'],keep='first').sort_values('TimeStamp').reset_index(drop=True)
assert 'PassOrFail' not in u and 'Reason' not in u
relations={
 'pressure':('Max_Injection_Pressure',['Max_Switch_Over_Pressure','Max_Injection_Speed','Filling_Time']),
 'fill':('Filling_Time',['Max_Injection_Pressure','Max_Injection_Speed','Mold_Temperature_3']),
 'cushion':('Cushion_Position',['Plasticizing_Position','Average_Back_Pressure','Barrel_Temperature_5']),
 'recovery':('Plasticizing_Time',['Max_Screw_RPM','Plasticizing_Position','Average_Back_Pressure'])}
# Online feature construction uses only past unlabeled sensor observations.
# Past unlabeled rows inside an evaluation interval can be used; no quality labels.
ut=u.TimeStamp.to_numpy(); features=np.full((len(d),4),np.nan);coverage=[]
for i,row in d.iterrows():
 stop=np.searchsorted(ut,row.TimeStamp.to_datetime64(),side='left')
 past=u.iloc[max(0,stop-100):stop].copy()
 if len(past):
  gap=past.TimeStamp.diff().dt.total_seconds()
  split=np.flatnonzero(gap.gt(1800).to_numpy())
  if len(split):past=past.iloc[split[-1]:]
 usable=len(past)>=20 and (row.TimeStamp-past.TimeStamp.iloc[-1]).total_seconds()<=1800
 coverage.append(dict(TimeStamp=row.TimeStamp,actual=int(y[i]),prior_unlabeled=len(past),usable=int(usable)))
 if not usable:continue
 assert past.TimeStamp.max()<row.TimeStamp
 for j,(name,(target,parents)) in enumerate(relations.items()):
  a=past[parents+[target]].dropna()
  if len(a)<20:continue
  sc=RobustScaler().fit(a[parents]);xp=sc.transform(a[parents])
  m=Ridge(alpha=1).fit(xp,a[target])
  residual=a[target].to_numpy()-m.predict(xp)
  scale=max(np.std(residual),float(a[target].std())*.1,1e-6)
  q=pd.DataFrame([row[parents].to_numpy(float)],columns=parents)
  features[i,j]=(float(row[target])-float(m.predict(sc.transform(q))[0]))/scale
print('Causal coverage',sum(z['usable'] for z in coverage),'of574; all8',sum(z['usable'] for z in coverage if z['actual']),flush=True)
views={'local_residual4':[0,1,2,3],'local_pressure2':[0,1],'local_supply2':[2,3]}
names=list(views);memo={}
def predict(ref,q):
 assert set(g[ref]).isdisjoint(g[q])
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(features[ref[y[ref]==0]])
 a=np.clip(imp.transform(features),-12,12);out=[]
 for name,cols in views.items():
  scores=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2)
    scores.append(m.fit(a[ref][:,cols],y[ref]).predict_proba(a[q][:,cols])[:,1])
  out.append(np.mean(scores,axis=0))
 result=np.column_stack(out);memo[key]=result;return result
weights=[(n,np.eye(3)[j]) for j,n in enumerate(names)]+[('local_expert_mean',np.ones(3)/3)]
def threshold(yy,s):
 p,r,c=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
 k=max(range(len(c)),key=lambda k:(f[k],r[k],c[k]));return float(c[k]),float(f[k])
records=[];choices=[]
for b in blocks:
 ref=np.flatnonzero(g!=b);q=np.flatnonzero(g==b);out=predict(ref,q)
 inner=np.full((len(ref),3),np.nan)
 for ib in np.unique(g[ref]):
  mask=g[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for name,w in weights:
  cal=inner@w;score=out@w;cut,fi=threshold(y[ref],cal);candidates.append((fi,name,w,cut))
  for rule,th in [('inner_F1',cut),('normal99',float(np.quantile(cal[y[ref]==0],.99,method='higher')))]:
   pr=score>=th
   for j,i in enumerate(q):records.append(dict(model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=th,prediction=int(pr[j])))
 fi,name,w,th=max(candidates,key=lambda z:(z[0],-np.count_nonzero(z[2]),z[1]));score=out@w;pr=score>=th
 choices.append(dict(block=b,selected=name,threshold=th,inner_F1=fi))
 for j,i in enumerate(q):records.append(dict(model='primary_inner_selected',rule='inner_F1',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=th,prediction=int(pr[j])))
 print('Completed causal teacher',b,name,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,rule),z in p.groupby(['model','rule']):
 assert len(z)==574 and z.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,rule=rule,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(z.actual,z.score)))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(coverage),'coverage'),(pd.DataFrame(choices),'choices')]:
 obj.to_csv(t/('rg3_shortshot_causal_fullhistory_'+name+'.csv'),index=False,encoding='utf-8-sig')
pd.DataFrame(features,columns=list(relations)).assign(TimeStamp=d.TimeStamp,actual=y).to_csv(t/'rg3_shortshot_causal_fullhistory_residuals.csv',index=False,encoding='utf-8-sig')
print(m.sort_values('F1',ascending=False).to_string(index=False))


Causal coverage 555 of574; all8 8


Completed causal teacher 20201021_001626_RG3 local_pressure2


Completed causal teacher 20201022_002405_RG3 local_pressure2


Completed causal teacher 20201022_044026_RG3 local_pressure2


Completed causal teacher 20201023_010128_RG3 local_residual4


Completed causal teacher 20201023_044148_RG3 local_pressure2


                 model     rule  TP  FP  FN  TN       F1       AP
       local_pressure2 inner_F1   3 175   5 391 0.032258 0.018197
primary_inner_selected inner_F1   2 117   6 449 0.031496 0.012167
     local_expert_mean inner_F1   8 559   0   7 0.027826 0.010748
       local_residual4 inner_F1   7 490   1  76 0.027723 0.010786
         local_supply2 inner_F1   7 519   1  47 0.026217 0.009601
       local_pressure2 normal99   0   2   8 564 0.000000 0.018197
     local_expert_mean normal99   0  10   8 556 0.000000 0.010748
       local_residual4 normal99   0  11   8 555 0.000000 0.010786
         local_supply2 normal99   0   2   8 564 0.000000 0.009601
